In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\03-agents\\02-agent-basics\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\03-agents\02-agent-basics


In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from minsearch import AppendableIndex

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [6]:

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [7]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

Traditional RAG

In [8]:
RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [9]:
##created a question with typo to test the RAG agent's ability to find the correct answer in the context
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [10]:

messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [11]:

print(response.output_text)

The provided context does not contain information on how to create a dashboard in Evidently.


because of the typo the RAG was not able to provide any answers


Agentic AI

In [11]:

#lets create a typo in the question to test the RAG agent's ability to find the correct answer in the context
question = "How do I create a dahsbrd in Evidnty?"

In [13]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: If you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


In [14]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

111

In [15]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidnty"}', call_id='call_eOSbGM4ebzVkug4IO2FtH5G7', name='search', type='function_call', id='fc_06c99a1cad9cc3130069f0031eed9481968c9933af318f9db0', namespace=None, status='completed')

In [16]:
tool_call = response.output[0]

In [17]:
messages.append(tool_call)

In [18]:
tool_call.arguments

'{"query":"create dashboard in Evidnty"}'

In [19]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidnty'}

In [20]:
# you can call the seach function in multiple ways, either by passing the query directly or by unpacking the arguments dictionary
#search_results = search(query='create dashboard in Evidently')
#search_results = search(query=arguments['query'])
search_results = search(**arguments)

In [21]:
search_results[:1]

[{'start': 0,
  'content': 'Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\n\n<Check>\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\n</Check>\n\n## Adding Tabs\n\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\n\n**To add a Tab**:\n\n- Enter "Edit" mode on the Dashboard (top right corner).\n- Click the plus sign with "add Tab" on the left.\n- To create a custom Tab, select "empty" and enter a name.\n\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\n\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\n\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, the Panels will appear empty un

In [22]:
## each tool call have a specific call id which you can use to send the output back to the model, this way the model can keep track of multiple tool calls and their outputs
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [23]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: If you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbrd in Evidnty?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidnty"}', call_id='call_eOSbGM4ebzVkug4IO2FtH5G7', name='search', type='function_call', id='fc_06c99a1cad9cc3130069f0031eed9481968c9933af318f9db0', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_eOSbGM4ebzVkug4IO2FtH5G7',
  'output': '[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\\n##

In [24]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

4029

In [25]:
print(response.output_text)

To create a dashboard in Evidently, follow these steps:

### 1. **Access the Dashboard**
- Navigate to your project and enter "Edit" mode by clicking the relevant button in the top right corner.

### 2. **Add Tabs** (Optional)
- Dashboards can have multiple tabs for organization.
- **To add a Tab**:
  - Click the plus sign for "add Tab" on the left.
  - Choose "empty" for a custom Tab and enter a name.

### 3. **Add Panels**
You can add various types of panels, including text panels, counters, pie charts, and line plots.

- **To add a Panel**:
  - Click on the "Add Panel" button next to the dashboard.
  - Follow the prompts to configure your panel (select metrics, set legends, etc.).
  - Use the preview feature to review your setup.
  - Click "Save" to add the Panel to your selected Tab.

### 4. **Configure the Panel**
- **Select Metrics**: Choose the corresponding Evidently Metric name that matches the names logged inside your Reports.
- **Filter by Tags & Metric Labels**: Use Tags to

In [26]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [27]:

response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

4253

In [28]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidnty, follow these steps:

### 1. **Add Reports**
   Ensure that you have added Reports with evaluation results to your Project. Dashboards display Panels that visualize these results over time.

### 2. **Create a Dashboard**
   - **User Interface (UI)**: 
     1. Navigate to the Dashboards section in Evidnty.
     2. Enter "Edit" mode on the Dashboard (top right corner).
     3. Click the "Add Panel" button to start adding visual components.

   - **Using Python API**: 
     ```python
     project.dashboard = ...  # Initialize or connect to your dashboard
     ```

### 3. **Adding Tabs** 
   - You can organize Panels across multiple Tabs. To add a Tab:
     - While in Edit mode, click the plus sign with "Add Tab".
     - Choose to create an empty Tab or select a pre-built template.

### 4. **Add Panels** 
   - Panels can represent various types of data visualizations like text, counters, pie charts, etc.
   - **To add a Panel in the UI**:
     1. Still in E